# Can we predict whether an email is spam?

This notebook builds and evaluates binary classifiers for the `Class` column in `data.csv`. We treat `Class = 1` as spam and `Class = 0` as non-spam.

The evaluation is designed to answer the practical question—not just maximize accuracy:

- compare learned models with a majority-class baseline;
- use a stratified, untouched 20% test set;
- report recall, precision, F1, ROC-AUC, and average precision;
- inspect errors and the most influential features.

**Important limitation:** this dataset contains engineered frequency/count features rather than raw email text. Results apply to emails represented in this same way.

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay, PrecisionRecallDisplay, RocCurveDisplay,
    accuracy_score, average_precision_score, balanced_accuracy_score,
    classification_report, confusion_matrix, f1_score, make_scorer, precision_score,
    recall_score, roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore', category=FutureWarning)
sns.set_theme(style='whitegrid', context='notebook')
RANDOM_STATE = 42

## 1. Load and validate the data

In [ ]:
DATA_PATH = Path('data.csv')
if not DATA_PATH.exists():
    raise FileNotFoundError(f'Expected {DATA_PATH.resolve()}')

df_raw = pd.read_csv(DATA_PATH)
assert 'Class' in df_raw.columns, "Expected a target column named 'Class'."
assert set(df_raw['Class'].dropna().unique()).issubset({0, 1}), 'Class must be binary (0/1).'

summary = pd.Series({
    'rows': len(df_raw),
    'predictors': df_raw.shape[1] - 1,
    'missing values': int(df_raw.isna().sum().sum()),
    'exact duplicate rows': int(df_raw.duplicated().sum()),
    'spam rows': int(df_raw['Class'].sum()),
    'spam rate': df_raw['Class'].mean(),
})
display(summary.to_frame('value'))
display(df_raw.head())

Exact duplicate rows are removed **before** splitting. Otherwise copies of the same observation could land in both train and test sets and make generalization look better than it is. We retain rows with identical predictors but conflicting labels, because they are not exact duplicates and reflect irreducible ambiguity in the data.

In [ ]:
df = df_raw.drop_duplicates().reset_index(drop=True)
X = df.drop(columns='Class')
y = df['Class'].astype(int)

non_numeric = X.select_dtypes(exclude=np.number).columns.tolist()
if non_numeric:
    raise TypeError(f'Unexpected non-numeric predictors: {non_numeric}')
if X.isna().any().any():
    raise ValueError('Missing predictor values require an imputation strategy.')

print(f'Rows used after exact deduplication: {len(df):,}')
print(f'Spam prevalence: {y.mean():.1%}')

## 2. Explore the target and feature relationships

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
counts = y.value_counts().sort_index()
axes[0].bar(['Non-spam (0)', 'Spam (1)'], counts.values, color=['#4C78A8', '#E45756'])
axes[0].set(title='Target distribution', ylabel='Emails')
for i, value in enumerate(counts.values):
    axes[0].text(i, value + 25, f'{value:,} ({value / len(y):.1%})', ha='center')

correlations = X.corrwith(y).sort_values()
strongest = pd.concat([correlations.head(6), correlations.tail(6)]).sort_values()
colors = ['#4C78A8' if v < 0 else '#E45756' for v in strongest.values]
axes[1].barh(strongest.index, strongest.values, color=colors)
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set(title='Strongest univariate correlations with spam', xlabel='Pearson correlation')
plt.tight_layout()
plt.show()

Correlation is descriptive, not causal, and it can miss nonlinear relationships. The predictive comparison below is the real test.

## 3. Create an untouched test set and define models

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

models = {
    'Dummy (majority class)': DummyClassifier(strategy='most_frequent'),
    'Logistic regression': Pipeline([
        ('scale', StandardScaler()),
        ('model', LogisticRegression(max_iter=3000, random_state=RANDOM_STATE)),
    ]),
    'Random forest': RandomForestClassifier(
        n_estimators=400, min_samples_leaf=2, n_jobs=-1, random_state=RANDOM_STATE
    ),
}

print(f'Training set: {len(X_train):,} rows; test set: {len(X_test):,} rows')
print(f'Train spam rate: {y_train.mean():.1%}; test spam rate: {y_test.mean():.1%}')

## 4. Cross-validation on the training data

Five-fold stratified cross-validation estimates model variability without touching the test set. ROC-AUC measures ranking across all thresholds; average precision is especially informative for the positive (spam) class.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    'accuracy': 'accuracy',
    'balanced_accuracy': 'balanced_accuracy',
    'precision': make_scorer(precision_score, zero_division=0),
    'recall': make_scorer(recall_score, zero_division=0),
    'f1': make_scorer(f1_score, zero_division=0),
    'roc_auc': 'roc_auc',
    'average_precision': 'average_precision',
}

cv_rows = []
for name, model in models.items():
    scores = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    row = {'model': name}
    for metric in scoring:
        values = scores[f'test_{metric}']
        row[metric] = values.mean()
        row[f'{metric}_sd'] = values.std()
    cv_rows.append(row)

cv_results = pd.DataFrame(cv_rows).set_index('model').sort_values('roc_auc', ascending=False)
display(cv_results[['accuracy', 'balanced_accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'average_precision']].style.format('{:.3f}'))

## 5. Final evaluation on the untouched test set

In [ ]:
test_rows = []
fitted_models = {}
test_predictions = {}
test_probabilities = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]
    fitted_models[name] = model
    test_predictions[name] = pred
    test_probabilities[name] = prob
    test_rows.append({
        'model': name,
        'accuracy': accuracy_score(y_test, pred),
        'balanced_accuracy': balanced_accuracy_score(y_test, pred),
        'precision': precision_score(y_test, pred, zero_division=0),
        'recall': recall_score(y_test, pred, zero_division=0),
        'f1': f1_score(y_test, pred, zero_division=0),
        'roc_auc': roc_auc_score(y_test, prob),
        'average_precision': average_precision_score(y_test, prob),
    })

test_results = pd.DataFrame(test_rows).set_index('model').sort_values('roc_auc', ascending=False)
display(test_results.style.format('{:.3f}'))

In [ ]:
learned_names = [name for name in models if not name.startswith('Dummy')]
best_name = test_results.loc[learned_names, 'roc_auc'].idxmax()
best_pred = test_predictions[best_name]
best_prob = test_probabilities[best_name]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
ConfusionMatrixDisplay.from_predictions(
    y_test, best_pred, display_labels=['Non-spam', 'Spam'], cmap='Blues', ax=axes[0], colorbar=False
)
axes[0].set_title(f'{best_name}: confusion matrix')
for name in learned_names:
    RocCurveDisplay.from_predictions(y_test, test_probabilities[name], name=name, ax=axes[1])
axes[1].plot([0, 1], [0, 1], '--', color='gray', label='Chance')
axes[1].set_title('ROC curves')
for name in learned_names:
    PrecisionRecallDisplay.from_predictions(y_test, test_probabilities[name], name=name, ax=axes[2])
axes[2].axhline(y_test.mean(), linestyle='--', color='gray', label='Spam prevalence')
axes[2].set_title('Precision–recall curves')
plt.tight_layout()
plt.show()

print(f'Classification report — {best_name} (threshold = 0.50)')
print(classification_report(y_test, best_pred, target_names=['Non-spam', 'Spam'], digits=3))

A false negative lets spam through; a false positive blocks legitimate email. The default 0.50 threshold is shown here, but a production threshold should be chosen from business costs using validation data—not by optimizing on this test set.

## 6. Which features matter to the best model?

In [ ]:
importance = permutation_importance(
    fitted_models[best_name], X_test, y_test, scoring='roc_auc',
    n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1,
)
importance_df = (
    pd.DataFrame({
        'feature': X.columns,
        'importance': importance.importances_mean,
        'std': importance.importances_std,
    })
    .sort_values('importance', ascending=False)
    .head(15)
    .sort_values('importance')
)

plt.figure(figsize=(9, 6))
plt.barh(importance_df['feature'], importance_df['importance'], xerr=importance_df['std'], color='#59A14F')
plt.xlabel('Decrease in test ROC-AUC after shuffling')
plt.title(f'Top permutation importances — {best_name}')
plt.tight_layout()
plt.show()
display(importance_df.sort_values('importance', ascending=False).reset_index(drop=True).style.format({'importance': '{:.4f}', 'std': '{:.4f}'}))

Permutation importance measures predictive contribution on held-out data, but correlated predictors can share importance. It does not establish that a feature causes spam.

## 7. Conclusion

In [ ]:
best = test_results.loc[best_name]
dummy = test_results.loc['Dummy (majority class)']
tn, fp, fn, tp = confusion_matrix(y_test, best_pred).ravel()
verdict = 'Yes' if best['roc_auc'] >= 0.80 and best['f1'] >= 0.70 else 'Only weakly'
display(Markdown(f'''
### {verdict}—these features predict spam well on a held-out sample.

The strongest evaluated model was **{best_name}**. On {len(y_test):,} previously unseen rows it achieved:

- **ROC-AUC:** {best['roc_auc']:.3f} (chance is 0.500)
- **Average precision:** {best['average_precision']:.3f} (spam prevalence baseline is {y_test.mean():.3f})
- **F1 for spam:** {best['f1']:.3f}
- **Spam precision / recall:** {best['precision']:.3f} / {best['recall']:.3f}
- **Accuracy:** {best['accuracy']:.3f}, versus {dummy['accuracy']:.3f} for always predicting the majority class
- **Errors at threshold 0.50:** {fp} legitimate emails flagged as spam; {fn} spam emails missed

This supports predictability for data drawn from the same process. Before deployment, validate on newer and genuinely out-of-domain email, monitor drift, select a threshold based on the relative cost of false positives and false negatives, and retrain periodically. The random row split cannot test time drift, sender/domain leakage, or performance on raw email text.
'''))